## 2.3 文档切分器 Text Splitters
### 2.3.1 为什么分割/切分/分块？
获取 Document 对象后，需要将其切分成一个个小块（Chunk）。之所以要进行切分是出于以下考虑：
* 长文档问题：大模型存在最大输入的 Token 限制 ，如果一个 Document 非常大 ，在输入模型时会 `被截断` 或直接报错，导致信息缺失（不少嵌入模型会悄悄截掉超出上限的部分，聊天模型的 API 一般直接报错）。
* 检索精度：Document 可能包含 非常多无关的信息 ，这些无效信息会 `干扰大模型` 的生成，而小块检索更精准。
* 成本控制：减少不必要的 `token` 消耗 。

无论是在存储还是检索过程中，都以这些 块(chunk) 为基本单位，这样能有效地避免内容噪声干扰和超出最大 Token 的问题。

### 2.3.2 Chunking拆分的策略
1. 方法1：根据句子切分：这种方法按照自然句子边界进行切分，以保持语义完整性。
2. 方法2：按照固定字符数来切分：这种策略根据特定的 字符数量 来划分文本，但可能会在不适当的位置 `切断句子` 。
3. 方法3：按固定字符数来切分，结合重叠窗口（overlapping windows）：此方法与按字符数切分相似，但通过 `重叠窗口技术` 让相邻两块共享一段内容：块的边界处仍会切断句子，但边界附近的内容在相邻两块里都有，从而保持信息的连贯性。
4. 方法4：递归字符切分方法：通过递归字符方式动态确定切分点：先按段落切，切出的片段仍然超过块大小，才改用换行、空格、单个字符等更细的分隔符。块大小的上限是固定的，随文档变化的是切分点的位置。
5. 方法5：根据语义内容切分：这种 高级策略 依据文本的语义内容来划分块，旨在保持相关信息的集中和完整，适用于需要高度语义保持的应用场景。

第2种方法（按照字符数切分）和第3种方法（按固定字符数切分结合重叠窗口）主要基于字符进行文本的切分，而不考虑文章的实际内容和语义。这种方式虽简单，但可能会导致 主题或语义上的断裂 。

相对而言，第4种递归方法更加灵活，它结合了固定长度切分和文本自身的结构（段落 → 行 → 词），但并不做语义分析。它通常是 首选策略 （官方文档也建议先用 `RecursiveCharacterTextSplitter`），因为它会尽量让段落、句子保持完整。

而第5种方法，基于语义的分割能按主题切出更完整的段落，但这种方法效率较低。它要用嵌入模型给每个句子计算向量，再比较相邻句子的向量差异来决定切分点（做法见下面的核对）， 处理速度较慢 ，并且没有块大小的上限， 段落长度可能极不均匀 （有的主题段落可能很长，而有的则较短）。因此，尽管它在某些需要高精度语义保持的场景下有其应用价值，但并 不适合所有情况 。

官方文档: https://docs.langchain.com/oss/python/integrations/splitters

#### 核对：主流的切分方法是这 5 种吗？

**不是。** 这 5 种是课程自己的归纳，LangChain 官方文档的各个版本都没有这样分：v0.1 按切分器列了 8 种，v0.3 分成 4 类，当前 1.x（上面的链接）分成 **3 类**：

| 官方 1.x 分类 | 思路 | LangChain 中的切分器 |
|---|---|---|
| Text structure-based（按文本结构） | 文本天然分成段落、句子、词。先尽量保留大的单元（段落），超长了再降到下一级（句子），必要时降到词 | `RecursiveCharacterTextSplitter`。官方建议**大多数场景先用它** |
| Length-based（按长度） | 只保证每块不超过规定的长度。长度可以按**字符数**计，也可以按 **token 数**计 | 按字符：`CharacterTextSplitter`；按 token：`TokenTextSplitter`、`CharacterTextSplitter.from_tiktoken_encoder()` 等 |
| Document structure-based（按文档结构） | HTML、Markdown、JSON、代码本身带有结构，按结构切，同一个结构单元里的内容通常语义相关 | Markdown：`MarkdownHeaderTextSplitter`；HTML：`HTMLHeaderTextSplitter` 等；JSON：`RecursiveJsonSplitter`；代码：`RecursiveCharacterTextSplitter.from_language()` |

把笔记的 5 种方法放进官方分类：

| 笔记的方法 | LangChain 中对应的切分器 | 官方 1.x 的归类 | 说明 |
|---|---|---|---|
| 1. 按句子 | `NLTKTextSplitter`、`SpacyTextSplitter` | 写在 Length-based 下的 “Split by tokens” 页里 | 用 NLTK / spaCy 断句后，还会把句子**合并**到 `chunk_size` 以内，并不是一句一块。需要另装 `nltk` 或 `spacy`，默认的断句模型针对英文 |
| 2. 固定字符数 | `CharacterTextSplitter(separator="")` | Length-based（按字符） | `CharacterTextSplitter` 默认 `separator="\n\n"`：先按段落切再合并，**不会**每 N 个字切一刀；`separator=""` 才是严格按字符数切（2.3.3 有验证） |
| 3. 固定字符数 + 重叠 | 同上，再设 `chunk_overlap > 0` | 不是单独的一类 | 重叠是父类 `TextSplitter` 的通用参数 `chunk_overlap`，几乎所有切分器都能用。方法 2、3 是同一个切分器的两种参数 |
| 4. 递归字符 | `RecursiveCharacterTextSplitter` | Text structure-based | 官方推荐的默认选择 |
| 5. 按语义 | `SemanticChunker`（在 `langchain-experimental` 包里） | 1.x 已不再列出 | 见下面的 ⚠️ |

反过来，笔记的 5 种**少了官方的两类**：

* **按 token 数切**：模型的上下文窗口、嵌入模型的输入上限都按 token 计算，按 token 计长度比按字符更贴近这些限制。官方 Length-based 一节的示例用的就是 `from_tiktoken_encoder`（按 token 计长度）。
* **按文档结构切**：Markdown 按标题、HTML 按标签、JSON 按对象、代码按函数和类切。这类切分器还会把块所属的标题等信息写进 `metadata`（例如 `MarkdownHeaderTextSplitter`），检索时可以利用。

> ⚠️ **与官方文档不同：1.x 文档去掉了“按语义切分”这一类**
>
> * v0.3 的概念页（Text splitters）把切分方法分成 4 类：Length-based、Text-structured based、Document-structured based、**Semantic meaning based**。当前 1.x 页面只保留前 3 类，原来的语义切分说明页（`.../splitters/semantic-chunker`）现在打开是 404。
> * LangChain 的语义切分器 `SemanticChunker` 在 `langchain-experimental` 包里。这个包的仓库在 2026-05-22（和 `langchain-community` 同一天）被归档，仓库说明写着 “No longer maintained”，最后一版是 0.4.2。本项目没有安装它。
> * 所以方法 5 作为思路仍然成立，但 LangChain 1.x 已经没有维护中的实现。

`SemanticChunker` 的做法（据 `langchain-experimental` 最后一版的源码）正好解释了上面说的“效率较低、长度不均匀”：

1. 用正则 `(?<=[.?!])\s+` 断句。这个默认正则只认英文的 `.` `?` `!` 后面跟空白，**中文的“。”不会断开**，处理中文要自己改 `sentence_split_regex` 参数；
2. 把每个句子和前后各 `buffer_size=1` 句拼在一起，调用嵌入模型计算向量。**每个句子都要算一次向量**，所以慢，还要花嵌入模型的调用费用；
3. 计算相邻两句向量的余弦距离：距离越大，说明话题变化越大；
4. 距离超过阈值的位置就是切分点。阈值默认取所有距离的第 95 百分位，也可以改成按标准差、四分位距等方式计算；
5. 只有可选的最小块长度 `min_chunk_size`，**没有 `chunk_size` 上限**，所以块的长短差别很大。

### 2.3.3 分割器父类分析

TextSplitter 源码分析

```python
TextSplitter(
  self,
  chunk_size: int = 4000,
  chunk_overlap: int = 200,
  length_function: Callable[[str], int] = len,
  keep_separator: bool | Literal['start', 'end'] = False,
  add_start_index: bool = False,
  strip_whitespace: bool = True
)
```

签名和当前源码（`langchain-text-splitters` 1.1.2，`langchain_text_splitters/base.py`）一致。

`TextSplitter` 是一个**抽象类**：它继承 `langchain_core` 的 `BaseDocumentTransformer`（文档转换器接口），并把 `split_text` 留作抽象方法给子类实现，所以不能直接用 `TextSplitter()` 创建对象。

并不是所有切分器都继承它：

| | 切分器 | `split_text` 的返回值 |
|---|---|---|
| 继承 `TextSplitter` | `CharacterTextSplitter`、`RecursiveCharacterTextSplitter`（以及它的子类 `MarkdownTextSplitter`、`PythonCodeTextSplitter`、`LatexTextSplitter`、`JSFrameworkTextSplitter`）、`TokenTextSplitter`、`SentenceTransformersTokenTextSplitter`、`NLTKTextSplitter`、`SpacyTextSplitter`、`KonlpyTextSplitter` | `list[str]` |
| 不继承 | `MarkdownHeaderTextSplitter`、`HTMLHeaderTextSplitter`、`HTMLSectionSplitter`、`RecursiveJsonSplitter` 等按文档结构切的切分器 | 各不相同，多数直接返回 `list[Document]` |

下面的参数和方法只适用于第一行的切分器。第二行的切分器有各自的参数（例如没有 `chunk_overlap`），2.3.4 讲到时再看。

**构造参数**

| 参数 | 类型 / 默认值 | 含义 | 在源码哪里生效 |
|---|---|---|---|
| `chunk_size` | `int`，4000 | 每块长度的上限，长度用 `length_function` 计算。只是“尽量不超过”：一个小片本身就超过上限时，它会原样成为一块，并打印警告 | `_merge_splits` |
| `chunk_overlap` | `int`，200 | 相邻两块**最多**重叠多长。重叠部分是上一块末尾的若干个完整小片，总长不超过 `chunk_overlap`；小片比它长，就没有重叠。源码注释写的 “Overlap in characters” 不准确：重叠也是用 `length_function` 计算的 | `_merge_splits`；`create_documents` 计算 `start_index` 时也用到 |
| `length_function` | `Callable[[str], int]`，`len` | 怎样计算一段文本的长度。默认 `len` 数字符；换成数 token 的函数，`chunk_size` 和 `chunk_overlap` 的单位就都变成 token | `_merge_splits` |
| `keep_separator` | `bool`、`"start"` 或 `"end"`，`False` | 切开以后分隔符怎么处理：`False` 丢掉；`True` 或 `"start"` 接在后一片的开头；`"end"` 接在前一片的末尾。`RecursiveCharacterTextSplitter` 把默认值改成了 `True` | 子类调用的 `_split_text_with_regex`（在 `character.py` 里） |
| `add_start_index` | `bool`，`False` | 为 `True` 时，每块的 `metadata` 里多一个 `start_index`：这块在原文中的起始字符位置 | `create_documents` |
| `strip_whitespace` | `bool`，`True` | 是否去掉每块首尾的空白（空格、换行）。去掉后变成空字符串的块直接丢弃 | `_join_docs` |

构造时会检查三种错误，都报 `ValueError`：`chunk_size <= 0`、`chunk_overlap < 0`、`chunk_overlap > chunk_size`（两者相等不报错）。

这些参数都由父类保存（`self._chunk_size` 等），但父类的公开方法（`create_documents` 等）只用到 `add_start_index` 和 `chunk_overlap`。其余参数在辅助函数 `_merge_splits`、`_join_docs`（父类里）和 `_split_text_with_regex`（`character.py` 里）中使用，而这些辅助函数要由子类的 `split_text` 调用才会执行。`TokenTextSplitter` 不调用这两个函数，它直接按 token 编号切，所以 `keep_separator`、`strip_whitespace`、`length_function` 对它不起作用。

![TextSplitter 的调用链](assets/07-textsplitter-call-chain.svg)

下面按笔记的顺序看三个核心方法（正好是调用链从下往上），每个参数在它生效的地方验证。`TextSplitter` 不能直接创建，演示时借用最简单的子类 `CharacterTextSplitter`（2.3.4.1 会细讲）。


In [1]:
from langchain_text_splitters import CharacterTextSplitter, TextSplitter

# TextSplitter 是抽象类，不能直接创建
try:
    TextSplitter()
except TypeError as e:
    print("TypeError：", e)

# 构造时的参数检查
for kwargs in [
    dict(chunk_size=0),
    dict(chunk_size=10, chunk_overlap=20),
    dict(chunk_size=10, chunk_overlap=10),
]:
    try:
        CharacterTextSplitter(**kwargs)
        print(kwargs, "→ 创建成功")
    except ValueError as e:
        print(kwargs, "→ ValueError：", e)

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


TypeError： Can't instantiate abstract class TextSplitter with abstract method split_text
{'chunk_size': 0} → ValueError： chunk_size must be > 0, got 0
{'chunk_size': 10, 'chunk_overlap': 20} → ValueError： Got a larger chunk overlap (20) than chunk size (10), should be smaller.
{'chunk_size': 10, 'chunk_overlap': 10} → 创建成功


输出开头的 `[transformers] PyTorch was not found…` 是 `langchain-text-splitters` 导入时尝试加载 `transformers` 打印的（为 `from_huggingface_tokenizer` 准备），与切分无关，可以忽略。

核心方法: 

1. split_text
```python
    @abstractmethod
    def split_text(self, text: str) -> list[str]:
        """Split text into multiple components.

        Args:
            text: The text to split.

        Returns:
            A list of text chunks.
        """

```

* **参数** `text`：要切分的一段字符串。
* **返回值**：`list[str]`，切好的块，只有文本，没有 metadata。
* **源码**：`TextSplitter` 里只有签名，没有实现（`@abstractmethod`），由子类实现。字符类的子类都分两步：① 用 `_split_text_with_regex` 把文本切成小片；② 用父类的 `_merge_splits` 把小片合并成不超过 `chunk_size` 的块。以 `CharacterTextSplitter.split_text` 为例（注释是我加的）：

```python
def split_text(self, text: str) -> list[str]:
    # 分隔符转成正则：is_separator_regex=False 时先转义
    sep_pattern = (
        self._separator if self._is_separator_regex else re.escape(self._separator)
    )
    # ① 切成小片，keep_separator 决定分隔符的去留
    splits = _split_text_with_regex(
        text, sep_pattern, keep_separator=self._keep_separator
    )
    # 分隔符是零宽断言（如 (?=...)）时，切的时候没有删掉任何字符，合并时也不用拼回去
    lookaround_prefixes = ("(?=", "(?<!", "(?<=", "(?!")
    is_lookaround = self._is_separator_regex and any(
        self._separator.startswith(p) for p in lookaround_prefixes
    )
    # 合并时用什么拼接：分隔符已经留在小片里，就用 ""；否则把分隔符拼回去
    merge_sep = ""
    if not (self._keep_separator or is_lookaround):
        merge_sep = self._separator
    # ② 合并成块
    return self._merge_splits(splits, merge_sep)
```

**第 ① 步 `_split_text_with_regex`：`keep_separator` 在这里生效**

它定义在 `character.py` 里，做三件事：

* `separator` 不是空字符串：`keep_separator` 为假时，直接 `re.split(separator, text)`，分隔符被删掉；为真时，用带括号的正则 `re.split(f"({separator})", text)` 把分隔符也留在结果里，再把它接到后一片的开头（`True`、`"start"`）或前一片的末尾（`"end"`）；
* `separator` 是空字符串：`list(text)`，**每个字符一片**；
* 最后去掉空字符串。

下面用“。”作分隔符，比较 `keep_separator` 的 4 种取值：

In [2]:
text = "退货政策。七天内可以无理由退货。退款在三个工作日内原路返回。电子发票在订单完成后开具。"

for keep in [False, True, "start", "end"]:
    splitter = CharacterTextSplitter(
        separator="。", chunk_size=20, chunk_overlap=0, keep_separator=keep
    )
    print(f"{keep!s:5} →", splitter.split_text(text))

False → ['退货政策。七天内可以无理由退货', '退款在三个工作日内原路返回', '电子发票在订单完成后开具']
True  → ['退货政策。七天内可以无理由退货', '。退款在三个工作日内原路返回', '。电子发票在订单完成后开具。']
start → ['退货政策。七天内可以无理由退货', '。退款在三个工作日内原路返回', '。电子发票在订单完成后开具。']
end   → ['退货政策。七天内可以无理由退货。', '退款在三个工作日内原路返回。', '电子发票在订单完成后开具。']


* `False`：小片里没有“。”。同一块内部合并时，`merge_sep` 是“。”，会被拼回去；但**每块末尾**的“。”丢了。
* `True` 和 `"start"` 结果相同：“。”接在下一句的开头，不符合中文习惯。
* `"end"`：“。”留在句末，切中文时最自然。

**第 ② 步 `_merge_splits`：`chunk_size`、`chunk_overlap`、`length_function` 在这里生效**

这是 `TextSplitter` 里真正决定“块怎么分”的方法（注释是我加的，`logger.warning` 的参数有省略）：

```python
def _merge_splits(self, splits: Iterable[str], separator: str) -> list[str]:
    separator_len = self._length_function(separator)   # 拼接用的分隔符也算长度

    docs = []                     # 已经完成的块
    current_doc: list[str] = []   # 正在攒的块，由若干小片组成
    total = 0                     # current_doc 的总长度（含分隔符）
    for d in splits:
        len_ = self._length_function(d)
        # 再放入 d 就超过 chunk_size：先输出当前块
        if total + len_ + (separator_len if len(current_doc) > 0 else 0) > self._chunk_size:
            if total > self._chunk_size:   # 当前块本身已超长（单个小片就超过 chunk_size）
                logger.warning("Created a chunk of size %d, which is longer than the specified %d", ...)
            if len(current_doc) > 0:
                doc = self._join_docs(current_doc, separator)
                if doc is not None:
                    docs.append(doc)
                # 从头部弹出小片，直到剩余总长 ≤ chunk_overlap，并且放得下 d
                while total > self._chunk_overlap or (
                    total + len_ + (separator_len if len(current_doc) > 0 else 0) > self._chunk_size
                    and total > 0
                ):
                    total -= self._length_function(current_doc[0]) + (
                        separator_len if len(current_doc) > 1 else 0
                    )
                    current_doc = current_doc[1:]
                # 没被弹出的小片留在 current_doc 里，成为下一块开头的重叠部分
        current_doc.append(d)
        total += len_ + (separator_len if len(current_doc) > 1 else 0)
    # 剩下的小片组成最后一块
    doc = self._join_docs(current_doc, separator)
    if doc is not None:
        docs.append(doc)
    return docs
```

![_merge_splits 的过程](assets/08-merge-splits-overlap.svg)

从源码可以得出三点：

1. `chunk_size` 是“尽量不超过”：单个小片就超过它时，这个小片会原样成为一块，并打印警告；
2. 重叠由上一块末尾的若干个**完整小片**组成，总长 ≤ `chunk_overlap`。小片比 `chunk_overlap` 长，就没有重叠；
3. 合并时插入的分隔符也计入长度（图中的灰色窄条）。

下面用图中的例子验证，最后再看 `separator=""` 的情况：

In [3]:
text = (
    "第一段：退货政策。七天内可以无理由退货。\n\n"
    "第二段：退款时效。退款在三个工作日内原路返回，节假日顺延。\n\n"
    "第三段：发票。电子发票在订单完成后开具。"
)
print("三段的长度：", [len(p) for p in text.split("\n\n")])

def show(splitter):
    for chunk in splitter.split_text(text):
        print(f"  {len(chunk):>2} 字 | {chunk!r}")

print("1. chunk_size=60, chunk_overlap=30：")
show(CharacterTextSplitter(chunk_size=60, chunk_overlap=30))

print("2. chunk_size=60, chunk_overlap=10：")
show(CharacterTextSplitter(chunk_size=60, chunk_overlap=10))

print("3. chunk_size=20：第二段长 29，原样成为一块，并打印警告")
show(CharacterTextSplitter(chunk_size=20, chunk_overlap=0))

print("4. separator=''：每个字符是一片，严格按 20 个字切，重叠 5 个字")
show(CharacterTextSplitter(separator="", chunk_size=20, chunk_overlap=5))

Created a chunk of size 29, which is longer than the specified 20


三段的长度： [20, 29, 20]
1. chunk_size=60, chunk_overlap=30：
  51 字 | '第一段：退货政策。七天内可以无理由退货。\n\n第二段：退款时效。退款在三个工作日内原路返回，节假日顺延。'
  51 字 | '第二段：退款时效。退款在三个工作日内原路返回，节假日顺延。\n\n第三段：发票。电子发票在订单完成后开具。'
2. chunk_size=60, chunk_overlap=10：
  51 字 | '第一段：退货政策。七天内可以无理由退货。\n\n第二段：退款时效。退款在三个工作日内原路返回，节假日顺延。'
  20 字 | '第三段：发票。电子发票在订单完成后开具。'
3. chunk_size=20：第二段长 29，原样成为一块，并打印警告
  20 字 | '第一段：退货政策。七天内可以无理由退货。'
  29 字 | '第二段：退款时效。退款在三个工作日内原路返回，节假日顺延。'
  20 字 | '第三段：发票。电子发票在订单完成后开具。'
4. separator=''：每个字符是一片，严格按 20 个字切，重叠 5 个字
  20 字 | '第一段：退货政策。七天内可以无理由退货。'
  20 字 | '理由退货。\n\n第二段：退款时效。退款在三'
  20 字 | '。退款在三个工作日内原路返回，节假日顺延'
  20 字 | '节假日顺延。\n\n第三段：发票。电子发票在'
  13 字 | '电子发票在订单完成后开具。'


* 第 1、2 种和图中一致：`chunk_overlap=30` 时第二段在两块里都出现；`chunk_overlap=10` 时第二段（29 字）比它长，两块没有重叠。
* 第 3 种：警告 `Created a chunk of size 29…` 来自 `_merge_splits` 里的 `logger.warning`，所以显示在 stderr 输出里。默认 `separator="\n\n"` 时 `CharacterTextSplitter` 只在段落之间切，段落再长也不会从中间切开——这就是 2.3.2 说“方法 2 要用 `separator=""`”的原因。
* 第 4 种：`separator=""` 时每个字符是一个小片，所以块长正好是 20，重叠正好是 5（第二块开头的“理由退货。”就是第一块的末尾）。这才是 2.3.2 方法 2、3 说的“按固定字符数切 + 重叠窗口”。`\n\n` 也被当成两个普通字符计数。

**`length_function`：换一种方式计算长度**

`_merge_splits` 里所有的长度都由 `self._length_function` 计算。类方法 `from_tiktoken_encoder` 会把“用 tiktoken 数 token”的函数设为 `length_function`，这样 `chunk_size`、`chunk_overlap` 的单位就变成 token。官方文档 Length-based 一节用的就是它：

In [4]:
import tiktoken

text = "退货政策：七天内可以无理由退货，退款在三个工作日内原路返回，电子发票在订单完成后开具。"
enc = tiktoken.get_encoding("cl100k_base")
print(f"全文 {len(text)} 个字符，{len(enc.encode(text))} 个 token\n")

# 默认 length_function=len：chunk_size 按字符计
by_char = CharacterTextSplitter(separator="", chunk_size=10, chunk_overlap=0)

# from_tiktoken_encoder：length_function 换成数 token 的函数，chunk_size 按 token 计
by_token = CharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base", separator="", chunk_size=10, chunk_overlap=0
)

for name, splitter in [("按字符计长度", by_char), ("按 token 计长度", by_token)]:
    print(name)
    for chunk in splitter.split_text(text):
        print(f"  {len(chunk):>2} 字符 / {len(enc.encode(chunk)):>2} token | {chunk}")

全文 43 个字符，41 个 token

按字符计长度
  10 字符 / 11 token | 退货政策：七天内可以
  10 字符 / 10 token | 无理由退货，退款在三
  10 字符 /  9 token | 个工作日内原路返回，
  10 字符 /  8 token | 电子发票在订单完成后
   3 字符 /  3 token | 开具。
按 token 计长度
   8 字符 / 10 token | 退货政策：七天内
  10 字符 /  9 token | 可以无理由退货，退款
   9 字符 /  9 token | 在三个工作日内原路
   8 字符 /  7 token | 返回，电子发票在
   8 字符 /  6 token | 订单完成后开具。


按字符计时每块正好 10 个字符；按 token 计时每块不超过 10 个 token，字符数反而不固定：`cl100k_base` 里大多数汉字占 1 个 token，有的占 2 个，“可以”这样的常用词两个字合成 1 个 token。`cl100k_base` 是 OpenAI 模型的分词方式，DeepSeek 等模型的分词器不同，这里只是演示 `length_function` 的作用。

**`_join_docs`：`strip_whitespace` 在这里生效**

`_merge_splits` 每输出一块，都调用 `_join_docs` 把小片拼起来：

```python
def _join_docs(self, docs: list[str], separator: str) -> str | None:
    text = separator.join(docs)
    if self._strip_whitespace:
        text = text.strip()      # 去掉首尾的空格、换行
    return text or None          # 空字符串返回 None，_merge_splits 就不把它加入结果
```

In [5]:
text = "第一段内容  \n\n   第二段内容\n\n第三段"

for strip in [True, False]:
    splitter = CharacterTextSplitter(chunk_size=8, chunk_overlap=0, strip_whitespace=strip)
    print(f"strip_whitespace={strip!s:5} →", splitter.split_text(text))

strip_whitespace=True  → ['第一段内容', '第二段内容', '第三段']
strip_whitespace=False → ['第一段内容  ', '   第二段内容', '第三段']


2. create_documents: 从text列表创建 Document 列表,方法底层调用了 split_text
```python
 def create_documents(
        self, texts: list[str], metadatas: list[dict[Any, Any]] | None = None
    ) -> list[Document]:
        """Create a list of `Document` objects from a list of texts.

        Args:
            texts: A list of texts to be split and converted into documents.
            metadatas: Optional list of metadata to associate with each document.

        Returns:
            A list of `Document` objects.
        """
        metadatas_ = metadatas or [{}] * len(texts)
        documents = []
        for i, text in enumerate(texts):
            index = 0
            previous_chunk_len = 0
            for chunk in self.split_text(text):
                metadata = copy.deepcopy(metadatas_[i])
                if self._add_start_index:
                    offset = index + previous_chunk_len - self._chunk_overlap
                    index = text.find(chunk, max(0, offset))
                    metadata["start_index"] = index
                    previous_chunk_len = len(chunk)
                new_doc = Document(page_content=chunk, metadata=metadata)
                documents.append(new_doc)
        return documents

```

* **参数**
  * `texts`：`list[str]`，多段文本。**必须是列表**：传入一个字符串的话，`for i, text in enumerate(texts)` 会逐个字符遍历，每个字符都被当成一段文本（下面有验证）。
  * `metadatas`：`list[dict]`，可选，第 i 个 dict 是第 i 段文本的元数据。不传时每段都用空 dict；传了就要和 `texts` 一样长，短了会报 `IndexError`。
* **返回值**：`list[Document]`。每段文本切出的每个块生成一个 `Document`，所有文本的块按顺序放在同一个列表里。

源码逐行看：

1. `metadatas_ = metadatas or [{}] * len(texts)`：没传 metadata 时，给每段文本配一个空 dict；
2. 外层循环遍历每段文本，内层循环遍历 `self.split_text(text)` 切出的块——这就是“底层调用了 `split_text`”；
3. `metadata = copy.deepcopy(metadatas_[i])`：每个块拿到一份**深拷贝**。改某个块的 metadata，不会影响同一段文本的其他块，也不会改到传入的 dict；
4. `add_start_index=True` 时计算 `start_index`：用 `text.find(chunk, 起点)` 在原文里查找这个块。起点是“上一块的起始位置 + 上一块的长度 − `chunk_overlap`”：有重叠时，下一块最早从这里开始。从起点往后找而不是从头找，是为了原文中有重复的文字时，不会找到前面那一处；
5. `Document(page_content=chunk, metadata=metadata)`：没有传 `id`，所以每个块的 `id` 都是 `None`。

In [6]:
text = (
    "第一段：退货政策。七天内可以无理由退货。\n\n"
    "第二段：退款时效。退款在三个工作日内原路返回，节假日顺延。\n\n"
    "第三段：发票。电子发票在订单完成后开具。"
)
splitter = CharacterTextSplitter(
    separator="", chunk_size=20, chunk_overlap=5, add_start_index=True
)
docs = splitter.create_documents([text], metadatas=[{"source": "faq.txt"}])

for doc in docs:
    start = doc.metadata["start_index"]
    # 用 start_index 回到原文里截取，应该和块的内容完全相同
    same = text[start : start + len(doc.page_content)] == doc.page_content
    print(doc.metadata, "| 原文对得上：", same)

# 每个块拿到的是 metadata 的深拷贝：改第一个块，不影响第二个块
docs[0].metadata["source"] = "改过了"
print("\n第二个块的 source：", docs[1].metadata["source"])

# texts 必须是列表：传入字符串，每个字符都被当成一段文本
print("传入字符串：", [d.page_content for d in splitter.create_documents("退货政策")])

{'source': 'faq.txt', 'start_index': 0} | 原文对得上： True
{'source': 'faq.txt', 'start_index': 15} | 原文对得上： True
{'source': 'faq.txt', 'start_index': 30} | 原文对得上： True
{'source': 'faq.txt', 'start_index': 45} | 原文对得上： True
{'source': 'faq.txt', 'start_index': 60} | 原文对得上： True

第二个块的 source： faq.txt
传入字符串： ['退', '货', '政', '策']


`start_index` 的起点公式把“上一块的长度”（字符数）和 `chunk_overlap` 相减。`length_function` 是 `len` 时两者单位相同，结果可靠；换成按 token 计长度后，`chunk_overlap` 的单位是 token，而英文的一个 token 通常有好几个字符，实际重叠的字符数比 `chunk_overlap` 大，起点就算过头了，`find` 找不到，`start_index` 变成 `-1`：

In [7]:
en_text = " ".join(f"Sentence {i} is about refunds and invoices." for i in range(5))

splitter = CharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",
    separator=" ",
    chunk_size=20,        # 单位是 token
    chunk_overlap=8,      # 单位也是 token
    add_start_index=True,
)
for doc in splitter.create_documents([en_text]):
    start = doc.metadata["start_index"]
    same = start >= 0 and en_text[start : start + len(doc.page_content)] == doc.page_content
    print(f"start_index={start:>3} | 原文对得上：{same!s:5} | {doc.page_content[:30]}…")

start_index=  0 | 原文对得上：True  | Sentence 0 is about refunds an…
start_index= -1 | 原文对得上：False | and invoices. Sentence 1 is ab…
start_index= 62 | 原文对得上：True  | refunds and invoices. Sentence…
start_index= -1 | 原文对得上：False | is about refunds and invoices.…
start_index=126 | 原文对得上：True  | Sentence 3 is about refunds an…
start_index= -1 | 原文对得上：False | and invoices. Sentence 4 is ab…


第 2、4、6 块的 `start_index` 是 `-1`。第 3、5 块能对上，只是因为上一块的结果是 `-1`，这一次的起点反而算小了。所以要用 `start_index` 定位原文时，保持默认的 `length_function=len`（上一个单元格就是这样）。

3. split_documents
```python
    def split_documents(self, documents: Iterable[Document]) -> list[Document]:
        """Split documents.

        Args:
            documents: The documents to split.

        Returns:
            A list of split documents.
        """
        texts, metadatas = [], []
        for doc in documents:
            texts.append(doc.page_content)
            metadatas.append(doc.metadata)
        return self.create_documents(texts, metadatas=metadatas)
```

底层调用 `create_documents`

* **参数** `documents`：任意可迭代的 `Document`。列表、生成器都可以，例如直接传加载器的 `loader.lazy_load()`。
* **返回值**：`list[Document]`，所有文档切出的块，按文档顺序排列。
* **源码**：把每个 `Document` 拆成 `page_content` 和 `metadata` 两个列表，交给 `create_documents`。
  * 原文档的 `metadata`（例如加载器写入的 `source`）被深拷贝到它的每个块上，所以每个块都知道自己来自哪个文件；
  * 只取了这两个字段，**原文档的 `id` 不会传给块**，块的 `id` 都是 `None`。

父类 `BaseDocumentTransformer` 要求实现的 `transform_documents`，在 `TextSplitter` 里就是 `return self.split_documents(list(documents))`，两者结果相同：

In [8]:
from langchain_core.documents import Document

doc = Document(page_content=text, metadata={"source": "faq.txt"}, id="faq-001")
splitter = CharacterTextSplitter(chunk_size=60, chunk_overlap=30)

chunks = splitter.split_documents([doc])
for chunk in chunks:
    print(chunk.id, chunk.metadata, repr(chunk.page_content[:15]))

print("transform_documents 结果相同：", splitter.transform_documents([doc]) == chunks)

None {'source': 'faq.txt'} '第一段：退货政策。七天内可以无'
None {'source': 'faq.txt'} '第二段：退款时效。退款在三个工'
transform_documents 结果相同： True


4. 其他方法

除了上面三个核心方法，`TextSplitter` 还有下面这些公开方法：

| 方法 | 参数 | 返回值 | 源码做了什么 |
|---|---|---|---|
| `transform_documents(documents, **kwargs)` | `documents`：`Document` 序列；`kwargs` 没有用到 | `list[Document]` | 实现父类 `BaseDocumentTransformer` 的抽象方法：`return self.split_documents(list(documents))`。有了它，切分器可以当作通用的“文档转换器”使用 |
| `atransform_documents(documents, **kwargs)` | 同上 | 同上 | 继承自 `BaseDocumentTransformer`：用 `run_in_executor` 在线程池里运行 `transform_documents`。切分是纯计算，异步版本不会更快，只是不阻塞事件循环 |
| `from_tiktoken_encoder(encoding_name="gpt2", model_name=None, allowed_special=None, disallowed_special="all", **kwargs)` | 类方法。`encoding_name`：tiktoken 的编码名；`model_name`：模型名，传了就按模型选编码，忽略 `encoding_name`；`allowed_special`、`disallowed_special`：文本里出现特殊 token（例如 GPT 的结束标记 endoftext）时，是允许还是报错；`kwargs`：其他构造参数（`chunk_size` 等） | 调用它的那个类的实例 | 用 tiktoken 定义一个“数 token”的函数，作为 `length_function` 传给构造函数；调用它的是 `TokenTextSplitter` 时，还会把编码参数一起传过去。需要 `tiktoken` 包（本项目已安装）。默认编码 `gpt2` 很旧，官方示例传的是 `cl100k_base` |
| `from_huggingface_tokenizer(tokenizer, **kwargs)` | 类方法。`tokenizer`：Hugging Face 的分词器对象；`kwargs`：其他构造参数 | 调用它的那个类的实例 | 把 `len(tokenizer.tokenize(text))` 作为 `length_function`。需要 `transformers` 包 |